Boilerplate Code

In [ ]:
#install if you dont already have the dependencies
pip install numpy pandas scikit-learn xgboost ucimlrepo matplotlib seaborn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#xgboost
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import (
    load_dataset,
    feature_types,
    split,
    make_preprocessor,
)
from src.corrupt import (
    inject_missing_mcar,
    inject_missing_mnar,
    flip_labels,
    perturb_features
)

split_seed = 42
experiment_seed = [42, 43, 44]

dataset_names = ["adult", "ai4i"]
corruption_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
noise_scale = 1.0

# 60 / 20 / 20 split
validation_split = 0.25 
#output directories
results_dir = ROOT / "results" / "xgboost"
models_dir = ROOT / "models" / "xgboost"
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)




certifications for downloading uci dataset

In [14]:
import socket
import os
import ssl
import urllib.request
import certifi

ssl_context = ssl.create_default_context(cafile=certifi.where())

opener = urllib.request.build_opener(
    urllib.request.HTTPSHandler(context=ssl_context)
)

urllib.request.install_opener(opener)

os.environ["SSL_CERT_FILE"] = certifi.where()
print(socket.gethostbyname("archive.ics.uci.edu"))

128.195.10.252


Load Data and create splits

In [15]:
clean_data = {}
split_sizes = []

for dataset_name in dataset_names:
    print(f"Loading dataset: {dataset_name}")
    X, y = load_dataset(dataset_name)

    # test split
    X_train_full, X_test, y_train_full, y_test = split(X, y, seed=split_seed)

    #val split
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=validation_split, stratify=y_train_full, random_state=split_seed)

    clean_data[dataset_name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    }

    split_sizes.append({
        "dataset": dataset_name,
        "train_size": len(X_train),
        "val_size": len(X_val),
        "test_size": len(X_test)
    })
pd.DataFrame(split_sizes)


Loading dataset: adult
Loading dataset: ai4i


,dataset,train_size,val_size,test_size
0,adult,24863,8288,8288
1,ai4i,6000,2000,2000


Tune Hyperparameters Using RandomSearchCV

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_grid = {
    "model__max_depth": [3, 4, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__reg_alpha": [0, 0.1, 1.0],
    "model__reg_lambda": [1, 5, 10],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=split_seed)

best_params = {}
searches = {}
search_results = []

for dataset_name in dataset_names:
    data = clean_data[dataset_name]
    
    pipeline = Pipeline([
        ("preprocess", make_preprocessor(data["X_train"], mode="native")),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            n_estimators=1000,
            missing=np.nan, # use NaN for missing values
            random_state=split_seed,
            n_jobs=-1, # use all available cores
        )),
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=30,
        scoring="average_precision",
        cv=cv,
        random_state=split_seed,
        n_jobs=1,
        refit=False,  # The training cell below will fit the selected configuration.
        verbose=2,
    )

    search.fit(data["X_train"], data["y_train"])
    searches[dataset_name] = search

    # get best parameters and remove the "model__" prefix
    best_params[dataset_name] = {
        key.replace("model__", ""): value for key, value in search.best_params_.items()
    }

    search_results.append({
        "dataset": dataset_name,
        "best_score": search.best_score_,
        "best_params": best_params[dataset_name],
    })

pd.DataFrame(search_results)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   2.0s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   2.0s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   2.3s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   2.0s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambd

,dataset,best_score,best_params
0,adult,0.844075,"{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha..."
1,ai4i,0.803442,"{'subsample': 0.7, 'reg_lambda': 5, 'reg_alpha..."


Train XGBoost

In [23]:
normal_models = {}
normal_preprocessors = {}
training_times = {}
for dataset_name in dataset_names:
    print(f"Training XGBoost model on dataset: {dataset_name}")
    data = clean_data[dataset_name]
    preprocessor = make_preprocessor(data["X_train"], mode="native") 

    X_train_preprocessed = preprocessor.fit_transform(data["X_train"])
    X_val_preprocessed = preprocessor.transform(data["X_val"])

    normal_preprocessors[dataset_name] = preprocessor

    for seed in experiment_seed:
        #define model
        # max rounds is 1000, early stopping is 30 consecutive rounds
        model = XGBClassifier( 
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            n_estimators=1000,
            early_stopping_rounds=30,
            missing=np.nan,
            random_state=seed,
            n_jobs=-1,
            **best_params[dataset_name],
        )
        # count triaining time
        start_time = time.perf_counter()

        model.fit(
            X_train_preprocessed, 
            data["y_train"], 
            eval_set=[(X_val_preprocessed, data["y_val"])],
            verbose=False
        )
        #log training time and save model
        training_times[(dataset_name, seed)] = time.perf_counter() - start_time
        normal_models[(dataset_name, seed)] = model
        print(f"Training time for {dataset_name} with seed {seed}: {training_times[(dataset_name, seed)]:.2f} seconds and best iteration: {model.best_iteration}")



Training XGBoost model on dataset: adult
Training time for adult with seed 42: 2.84 seconds and best iteration: 615
Training time for adult with seed 43: 2.80 seconds and best iteration: 615
Training time for adult with seed 44: 2.79 seconds and best iteration: 615
Training XGBoost model on dataset: ai4i
Training time for ai4i with seed 42: 0.80 seconds and best iteration: 273
Training time for ai4i with seed 43: 0.51 seconds and best iteration: 165
Training time for ai4i with seed 44: 0.61 seconds and best iteration: 209


Validation

In [24]:
from sklearn.metrics import average_precision_score

normal_val_metrics = []

for dataset_name in dataset_names:
    data = clean_data[dataset_name]
    preprocessor = normal_preprocessors[dataset_name]

    X_val_preprocessed = preprocessor.transform(data["X_val"])
    y_val = data["y_val"]

    for seed in experiment_seed:
        model = normal_models[(dataset_name, seed)]

        y_pred = model.predict(X_val_preprocessed)
        y_score = model.predict_proba(X_val_preprocessed)[:, 1]

        normal_val_metrics.append({
            "dataset": dataset_name,
            "seed": seed,
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, zero_division=0),
            "recall": recall_score(y_val, y_pred, zero_division=0),
            "f1": f1_score(y_val, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_val, y_score),
            "best_round": model.best_iteration + 1,
            "training_seconds": training_times[(dataset_name, seed)],
        })

normal_val_df = pd.DataFrame(normal_val_metrics)
normal_val_df.round(4)

,dataset,seed,accuracy,precision,recall,f1,average_precision,best_round,training_seconds
0,adult,42,0.8837,0.8054,0.6779,0.7362,0.8486,616,2.8398
1,adult,43,0.8837,0.8054,0.6779,0.7362,0.8486,616,2.8018
2,adult,44,0.8837,0.8054,0.6779,0.7362,0.8486,616,2.7925
3,ai4i,42,0.9805,0.8537,0.5147,0.6422,0.7558,274,0.7986
4,ai4i,43,0.9805,0.8718,0.5000,0.6355,0.7437,166,0.5104
5,ai4i,44,0.9810,0.8947,0.5000,0.6415,0.7580,210,0.6078


Test

In [25]:
normal_test_metrics = []

for dataset_name in dataset_names:
    data = clean_data[dataset_name]
    preprocessor = normal_preprocessors[dataset_name]

    # Apply the existing preprocessing; do not fit it again.
    X_test_preprocessed = preprocessor.transform(data["X_test"])
    y_test = data["y_test"]

    for seed in experiment_seed:
        model = normal_models[(dataset_name, seed)]

        y_pred = model.predict(X_test_preprocessed)
        y_score = model.predict_proba(X_test_preprocessed)[:, 1]

        normal_test_metrics.append({
            "dataset": dataset_name,
            "seed": seed,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, y_score),
        })

normal_test_df = pd.DataFrame(normal_test_metrics)
normal_test_df.round(4)

,dataset,seed,accuracy,precision,recall,f1,average_precision
0,adult,42,0.8855,0.8014,0.6935,0.7436,0.8541
1,adult,43,0.8855,0.8014,0.6935,0.7436,0.8541
2,adult,44,0.8855,0.8014,0.6935,0.7436,0.8541
3,ai4i,42,0.9840,0.8750,0.6176,0.7241,0.8086
4,ai4i,43,0.9815,0.8298,0.5735,0.6783,0.7957
5,ai4i,44,0.9835,0.8571,0.6176,0.7179,0.8025


Performance for test and validation was about the same so no underfitting or overfitting clearly evident. This is the control values that we will be using to test the models performance against missing data. 

In [ ]:
corruption_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
corruption_seed = 42
corrupted_results_dir = results_dir / "mcar"
corrupted_results_dir.mkdir(parents=True, exist_ok=True)

param_grid = {
    "model__max_depth": [3, 4, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__reg_alpha": [0, 0.1, 1.0],
    "model__reg_lambda": [1, 5, 10],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=split_seed)

corrupted_best_params = {}
corrupted_searches = {}
corrupted_search_results = []
corrupted_models = {}
corrupted_preprocessors = {}
corrupted_training_times = {}
corrupted_val_metrics = []
corrupted_test_metrics = []
missingness = []

for dataset_name in dataset_names:
    for missing_rate in corruption_rates:
        print(f"Training XGBoost on {dataset_name} with {missing_rate:.0%} missing data")

        # use the same splits as the normal models
        data = clean_data[dataset_name].copy()

        # add missing values to the training and validation sets
        data["X_train"] = inject_missing_mcar(
            data["X_train"], missing_rate,
            rng=np.random.default_rng(corruption_seed)
        )
        data["X_val"] = inject_missing_mcar(
            data["X_val"], missing_rate,
            rng=np.random.default_rng(corruption_seed + 1)
        )

        #check missingness
        missingness.append({
            "dataset": dataset_name,
            "missing_rate": missing_rate,
            "corruption_seed": corruption_seed,
            "training_missing_pct": 100 * data["X_train"].isna().to_numpy().mean(),
            "validation_missing_pct": 100 * data["X_val"].isna().to_numpy().mean(),
            "test_missing_pct": 100 * data["X_test"].isna().to_numpy().mean(),
        })

        # Tune Hyperparameters Using RandomSearchCV
        pipeline = Pipeline([
            ("preprocess", make_preprocessor(data["X_train"], mode="native")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                n_estimators=1000,
                missing=np.nan, # use NaN for missing values
                random_state=split_seed,
                n_jobs=-1, # use all available cores
            )),
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            n_iter=30,
            scoring="average_precision",
            cv=cv,
            random_state=split_seed,
            n_jobs=1,
            refit=False,  # The training code below will fit the selected configuration.
            verbose=2,
        )

        start_time = time.perf_counter()
        search.fit(data["X_train"], data["y_train"])
        corrupted_searches[(dataset_name, missing_rate)] = search

        # get best parameters and remove the "model__" prefix
        corrupted_best_params[(dataset_name, missing_rate)] = {
            key.replace("model__", ""): value for key, value in search.best_params_.items()
        }
        corrupted_search_results.append({
            "dataset": dataset_name,
            "missing_rate": missing_rate,
            "corruption_seed": corruption_seed,
            "best_score": search.best_score_,
            "best_params": corrupted_best_params[(dataset_name, missing_rate)],
            "tuning_seconds": time.perf_counter() - start_time,
        })

        # Train XGBoost
        preprocessor = make_preprocessor(data["X_train"], mode="native")

        X_train_preprocessed = preprocessor.fit_transform(data["X_train"])
        X_val_preprocessed = preprocessor.transform(data["X_val"])

        corrupted_preprocessors[(dataset_name, missing_rate)] = preprocessor

        for seed in experiment_seed:
            #define model
            # max rounds is 1000, early stopping is 30 consecutive rounds
            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                n_estimators=1000,
                early_stopping_rounds=30,
                missing=np.nan,
                random_state=seed,
                n_jobs=-1,
                **corrupted_best_params[(dataset_name, missing_rate)],
            )
            # count training time
            start_time = time.perf_counter()

            model.fit(
                X_train_preprocessed,
                data["y_train"],
                eval_set=[(X_val_preprocessed, data["y_val"])],
                verbose=False
            )
            #log training time and save model
            corrupted_training_times[(dataset_name, missing_rate, seed)] = time.perf_counter() - start_time
            corrupted_models[(dataset_name, missing_rate, seed)] = model
            print(f"Training time for {dataset_name} with {missing_rate:.0%} missing data and seed {seed}: {corrupted_training_times[(dataset_name, missing_rate, seed)]:.2f} seconds and best iteration: {model.best_iteration}")

        # Validation
        y_val = data["y_val"]

        for seed in experiment_seed:
            model = corrupted_models[(dataset_name, missing_rate, seed)]

            y_pred = model.predict(X_val_preprocessed)
            y_score = model.predict_proba(X_val_preprocessed)[:, 1]

            corrupted_val_metrics.append({
                "dataset": dataset_name,
                "missing_rate": missing_rate,
                "corruption_seed": corruption_seed,
                "seed": seed,
                "accuracy": accuracy_score(y_val, y_pred),
                "precision": precision_score(y_val, y_pred, zero_division=0),
                "recall": recall_score(y_val, y_pred, zero_division=0),
                "f1": f1_score(y_val, y_pred, zero_division=0),
                "average_precision": average_precision_score(y_val, y_score),
                "best_round": model.best_iteration + 1,
                "training_seconds": corrupted_training_times[(dataset_name, missing_rate, seed)],
            })

        # Test
        # Apply the existing preprocessing; do not fit it again.
        X_test_preprocessed = preprocessor.transform(data["X_test"])
        y_test = data["y_test"]

        for seed in experiment_seed:
            model = corrupted_models[(dataset_name, missing_rate, seed)]

            y_pred = model.predict(X_test_preprocessed)
            y_score = model.predict_proba(X_test_preprocessed)[:, 1]

            corrupted_test_metrics.append({
                "dataset": dataset_name,
                "missing_rate": missing_rate,
                "corruption_seed": corruption_seed,
                "seed": seed,
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1": f1_score(y_test, y_pred, zero_division=0),
                "average_precision": average_precision_score(y_test, y_score),
                "best_round": model.best_iteration + 1,
                "training_seconds": corrupted_training_times[(dataset_name, missing_rate, seed)],
            })

        #save results after each missing rate
        corrupted_val_df = pd.DataFrame(corrupted_val_metrics)
        corrupted_test_df = pd.DataFrame(corrupted_test_metrics)
        corrupted_val_df.to_csv(corrupted_results_dir / "validation.csv", index=False)
        corrupted_test_df.to_csv(corrupted_results_dir / "test.csv", index=False)
        pd.DataFrame(corrupted_search_results).to_csv(corrupted_results_dir / "tuning.csv", index=False)
        pd.DataFrame(missingness).to_csv(corrupted_results_dir / "missingness.csv", index=False)
        metrics_df = pd.concat([
            corrupted_val_df.assign(partition="validation"),
            corrupted_test_df.assign(partition="test"),
        ], ignore_index=True)
        metrics_df.to_csv(corrupted_results_dir / "metrics.csv", index=False)

#average test performance across the three seeds
test_summary = corrupted_test_df.groupby(["dataset", "missing_rate"])[
    ["accuracy", "precision", "recall", "f1", "average_precision"]
].agg(["mean", "std"])

test_summary.to_csv(corrupted_results_dir / "test_summary.csv")
display(test_summary.round(4))

Training XGBoost on adult with 10% missing data
Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   4.0s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   3.4s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   2.9s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_weight=3, model__reg_alpha=0.1, model__reg_lambda=10, model__subsample=0.7; total time=   3.1s
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=4, model__min_child_

accuracy         precision          recall          \
                         mean     std      mean     std    mean     std   
dataset missing_rate                                                      
adult   0.1            0.8862  0.0000    0.7993  0.0000  0.7006  0.0000   
        0.2            0.8861  0.0000    0.7975  0.0000  0.7026  0.0000   
        0.3            0.8838  0.0000    0.7899  0.0000  0.7011  0.0000   
        0.4            0.8801  0.0000    0.7809  0.0000  0.6935  0.0000   
        0.5            0.8781  0.0000    0.7579  0.0000  0.7213  0.0000   
ai4i    0.1            0.9800  0.0015    0.8495  0.0288  0.5000  0.0294   
        0.2            0.9777  0.0020    0.7857  0.0143  0.4706  0.0674   
        0.3            0.9750  0.0005    0.7548  0.0111  0.3922  0.0170   
        0.4            0.9685  0.0005    0.6891  0.0626  0.1373  0.0085   
        0.5            0.9667  0.0006    0.4306  0.3735  0.0441  0.0389   

                          f1         average_precision          
                        mean     std              mean     std  
dataset missing_rate                                            
adult   0.1           0.7467  0.0000            0.8546  0.0000  
        0.2           0.7471  0.0000            0.8540  0.0000  
        0.3           0.7429  0.0000            0.8524  0.0000  
        0.4           0.7347  0.0000            0.8460  0.0000  
        0.5           0.7392  0.0000            0.8405  0.0000  
ai4i    0.1           0.6294  0.0312            0.7694  0.0103  
        0.2           0.5871  0.0575            0.7160  0.0196  
        0.3           0.5160  0.0152            0.6631  0.0107  
        0.4           0.2284  0.0085            0.5891  0.0099  
        0.5           0.0799  0.0702            0.4407  0.0481

Results saved to: c:\Users\Lenovo P52\ECS Classes\EEC 171\ECS-171-Project\results\xgboost\mcar


Missing values were added to the training and validaiton features while leaving the test set unchanged. For these datasets, it is more accurate to look at precision since there is already a high chance of accuracy to the data being very skewed. We can see that precision for the adult income is around the same even with more missing data. This shows XGBoosts performance with missing data is relatively the same as we increase the percentage of missingness for the adult dataset. The same is not true for the ai4i set. Average_precision drop from 0.76 to 0.44 as we increase from 10% missingness to 50% missingness. This could be due to how heavily skewed the ai4i dataset is with only 3.4% of machines failing. With more data missing, it makes it harder for the model to determine what actual features have the greatest impact on a machine failing. 